In [0]:
df=spark.read.csv("/Volumes/workspace/dataframe/dataframe/netflix_titles.csv", header=True, inferSchema=True)
df.printSchema()
df.show()

root
 |-- show_id: string (nullable = true)
 |-- type: string (nullable = true)
 |-- title: string (nullable = true)
 |-- director: string (nullable = true)
 |-- cast: string (nullable = true)
 |-- country: string (nullable = true)
 |-- date_added: string (nullable = true)
 |-- release_year: string (nullable = true)
 |-- rating: string (nullable = true)
 |-- duration: string (nullable = true)
 |-- listed_in: string (nullable = true)
 |-- description: string (nullable = true)

+-------+-------+--------------------+--------------------+--------------------+--------------------+------------------+------------+------+---------+--------------------+--------------------+
|show_id|   type|               title|            director|                cast|             country|        date_added|release_year|rating| duration|           listed_in|         description|
+-------+-------+--------------------+--------------------+--------------------+--------------------+------------------+------------+

In [0]:
#Count of Movies vs TV Shows.
Movie_count = df.select("type").where("type == 'Movie'").count()
TV_show_count = df.select("type").where("type == 'TV Show'").count()
print(f"Movie vs TVShow count is Movie {Movie_count} vs TVShow {TV_show_count}")


Movie vs TVShow count is Movie 6131 vs TVShow 2676


In [0]:
#Count of null/missing `director` values.
null_director = df.select("director").where("director is null").count()
print(f"Null director count is {null_director}")


Null director count is 2636


In [0]:
#Count of titles per `rating`, sorted descending.



title_count = df.groupBy('rating').count().withColumnRenamed('count', 'title_count').orderBy('title_count', ascending=False)
title_count.show()

+------------+-----------+
|      rating|title_count|
+------------+-----------+
|       TV-MA|       3195|
|       TV-14|       2158|
|       TV-PG|        862|
|           R|        796|
|       PG-13|        489|
|       TV-Y7|        334|
|        TV-Y|        307|
|          PG|        286|
|        TV-G|        220|
|          NR|         80|
|           G|         41|
|        NULL|          6|
|    TV-Y7-FV|          6|
|          UR|          3|
|       NC-17|          3|
|        2021|          2|
|        2019|          1|
|        2017|          1|
| Jide Kosoko|          1|
|        2006|          1|
+------------+-----------+
only showing top 20 rows


In [0]:
#Top 10 `listed_in` genres by title count
top_genres = df.groupBy('listed_in').count().withColumnRenamed('count', 'title_count').orderBy('title_count', ascending=False).limit(10)
top_genres.show()

+--------------------+-----------+
|           listed_in|title_count|
+--------------------+-----------+
|Dramas, Internati...|        361|
|       Documentaries|        358|
|     Stand-Up Comedy|        334|
|Comedies, Dramas,...|        273|
|Dramas, Independe...|        252|
|            Kids' TV|        220|
|Children & Family...|        215|
|Children & Family...|        201|
|Documentaries, In...|        186|
|Dramas, Internati...|        180|
+--------------------+-----------+



In [0]:
#Top 15 genres by title count
top_genres_count = df.groupBy('listed_in').count().withColumnRenamed('count', 'title_count').orderBy('title_count', ascending=False).limit(15)
top_genres.show()


+--------------------+-----------+
|           listed_in|title_count|
+--------------------+-----------+
|Dramas, Internati...|        361|
|       Documentaries|        358|
|     Stand-Up Comedy|        334|
|Comedies, Dramas,...|        273|
|Dramas, Independe...|        252|
|            Kids' TV|        220|
|Children & Family...|        215|
|Children & Family...|        201|
|Documentaries, In...|        186|
|Dramas, Internati...|        180|
+--------------------+-----------+



In [0]:
#Type of available content
#Unique content types and their counts
content = df.groupBy("type").count().orderBy("count", ascending=False)
content.show()

# Unique genres count
unique_genres_count = df.select("listed_in").distinct().count()
print(f"Total unique genres/categories: {unique_genres_count}")

+-------------+-----+
|         type|count|
+-------------+-----+
|        Movie| 6131|
|      TV Show| 2676|
|         NULL|    1|
|William Wyler|    1|
+-------------+-----+

Total unique genres/categories: 534


In [0]:
#Quality of newly added contents by month and year.
from pyspark.sql.functions import try_to_date, year, month, trim, col

# Parse date_added and extract year/month
df_with_dates = df.withColumn("added_date", try_to_date(trim(col("date_added")), "MMMM d, yyyy")) \
                  .withColumn("added_year", year(col("added_date"))) \
                  .withColumn("added_month", month(col("added_date")))

# Filter for content added after 2020
df_filtered = df_with_dates.filter(col("added_year") > 2020)

# Quality of newly added contents by month and year
quality_by_period = df_filtered.groupBy("added_year", "added_month", "rating").count() \
                               .orderBy("count", ascending=False)
quality_by_period.show(100, truncate=False)


+----------+-----------+------+-----+
|added_year|added_month|rating|count|
+----------+-----------+------+-----+
|2021      |6          |TV-MA |85   |
|2021      |7          |TV-MA |78   |
|2021      |4          |TV-MA |64   |
|2021      |8          |TV-MA |58   |
|2021      |6          |TV-14 |51   |
|2021      |7          |TV-14 |50   |
|2021      |5          |TV-MA |50   |
|2021      |8          |TV-14 |46   |
|2021      |2          |TV-MA |44   |
|2021      |9          |TV-MA |44   |
|2021      |4          |TV-14 |41   |
|2021      |3          |TV-MA |34   |
|2021      |7          |R     |33   |
|2021      |2          |TV-14 |31   |
|2021      |7          |PG-13 |30   |
|2021      |1          |R     |29   |
|2021      |1          |TV-MA |29   |
|2021      |9          |TV-14 |28   |
|2021      |9          |PG-13 |28   |
|2021      |3          |TV-14 |27   |
|2021      |7          |TV-Y7 |26   |
|2021      |1          |TV-14 |26   |
|2021      |5          |TV-14 |25   |
|2021      |

In [0]:
#Average duration of Tv Shows in seasons
from pyspark.sql.functions import col, expr, avg

# Filter for TV Shows only
df_tv = df.filter(col("type") == "TV Show")

# Extract numeric duration from strings like "1 Season", "2 Seasons"
# Using try_cast to handle empty strings gracefully (returns NULL instead of error)
df_tv = df_tv.withColumn(
    "duration_num",
    expr("try_cast(regexp_extract(duration, '(\\\\d+)', 1) as int)")
)

# Calculate average number of seasons
avg_duration = df_tv.agg(avg("duration_num").alias("avg_duration"))

print("Average duration of TV Shows:")
avg_duration.show()


Average duration of TV Shows:
+------------------+
|      avg_duration|
+------------------+
|1.7700074794315632|
+------------------+



In [0]:
#Its Christmas! What movies/Tv shows are you binge watching this weekend?
from pyspark.sql.functions import col, lower

# Filter for Christmas-themed content in title or description
christmas_content = df.filter(
    lower(col("title")).contains("christmas") | 
    lower(col("description")).contains("christmas") |
    lower(col("title")).contains("holiday") |
    lower(col("title")).contains("xmas")
)

# Count by type
print("Christmas Content Summary:")
christmas_content.groupBy("type").count().orderBy("count", ascending=False).show()

print("\nChristmas Movies and TV Shows:")
christmas_content.select("type", "title", "release_year", "rating", "duration") \
    .orderBy(col("release_year").desc(), "type", "title") \
    .show(50, truncate=False)


Christmas Content Summary:
+-------+-----+
|   type|count|
+-------+-----+
|  Movie|  108|
|TV Show|   14|
+-------+-----+


Christmas Movies and TV Shows:
+-------+------------------------------------------+------------+------+---------+
|type   |title                                     |release_year|rating|duration |
+-------+------------------------------------------+------------+------+---------+
|Movie  |A California Christmas                    |2020        |PG-13 |107 min  |
|Movie  |A Go! Go! Cory Carson Christmas           |2020        |TV-Y  |22 min   |
|Movie  |A New York Christmas Wedding              |2020        |TV-MA |90 min   |
|Movie  |A Trash Truck Christmas                   |2020        |TV-Y  |28 min   |
|Movie  |Alien Xmas                                |2020        |TV-Y  |42 min   |
|Movie  |An Unremarkable Christmas                 |2020        |TV-14 |83 min   |
|Movie  |Angela's Christmas Wish                   |2020        |TV-Y  |48 min   |
|Movie  |Capta

In [0]:
#My friend is a Ryan Reynold fan , I am thinking of making a movie compilation. What movies should I include ?
from pyspark.sql.functions import col, lower

# Filter for Ryan Reynolds in the cast
ryan_reynolds_content = df.filter(
    lower(col("cast")).contains("ryan reynolds")
)

# Count by type
print("Ryan Reynolds Content Summary:")
ryan_reynolds_content.groupBy("type").count().orderBy("count", ascending=False).show()

print("\nRyan Reynolds Movies and TV Shows:")
ryan_reynolds_content.select("type", "title", "release_year", "rating", "duration", "listed_in") \
    .orderBy(col("release_year").desc(), "title") \
    .show(50, truncate=False)

Ryan Reynolds Content Summary:
+-----+-----+
| type|count|
+-----+-----+
|Movie|   13|
+-----+-----+


Ryan Reynolds Movies and TV Shows:
+-----+-------------------------------------+------------+------+--------+------------------------------------+
|type |title                                |release_year|rating|duration|listed_in                           |
+-----+-------------------------------------+------------+------+--------+------------------------------------+
|Movie|6 Underground                        |2019        |R     |129 min |Action & Adventure, Dramas          |
|Movie|Betty White: First Lady of Television|2018        |TV-14 |56 min  |Documentaries                       |
|Movie|Mississippi Grind                    |2015        |R     |109 min |Dramas, Independent Movies          |
|Movie|Selfless                             |2015        |PG-13 |117 min |Sci-Fi & Fantasy, Thrillers         |
|Movie|The Captive                          |2014        |R     |112 min |Dram

In [0]:
#How many movies or shows were released during COVID season.
from pyspark.sql.functions import expr

movies_in_covid = df.select("release_year").where(expr("try_cast(release_year as int) between 2020 and 2021"))
movies_in_covid.count()

shows = df.groupBy("type").count().orderBy("count", ascending=False)
shows.show()

+-------------+-----+
|         type|count|
+-------------+-----+
|        Movie| 6131|
|      TV Show| 2676|
|         NULL|    1|
|William Wyler|    1|
+-------------+-----+



In [0]:
# I am on a short flight of 2 hours, which movies or shows I should download to watch on the flight ?
from pyspark.sql.functions import col, expr, regexp_extract


short_movies = df.filter(col("type") == "Movie") \
    .withColumn("duration_mins", expr("try_cast(regexp_extract(duration, '(\\\\d+)', 1) as int)")) \
    .filter(col("duration_mins") < 120) \
    .filter(col("duration_mins").isNotNull())

print(f"Found {short_movies.count()} movies under 2 hours (120 minutes)")

print("\nRecommended Movies for Your Flight:")
short_movies.select("title", "duration", "duration_mins", "rating", "release_year", "listed_in") \
    .orderBy(col("duration_mins").desc()) \
    .show(30, truncate=False)


Found 4919 movies under 2 hours (120 minutes)

Recommended Movies for Your Flight:
+-----------------------------------------+--------+-------------+------+------------+----------------------------------------------------------+
|title                                    |duration|duration_mins|rating|release_year|listed_in                                                 |
+-----------------------------------------+--------+-------------+------+------------+----------------------------------------------------------+
|Surga Yang Tak Dirindukan 2              |119 min |119          |TV-14 |2017        |Dramas, Faith & Spirituality, International Movies        |
|Savita Damodar Paranjpe                  |119 min |119          |TV-14 |2018        |Horror Movies, International Movies                       |
|Monsters: Dark Continent                 |119 min |119          |R     |2014        |Action & Adventure, Dramas, Independent Movies            |
|Uyare                                   